In [12]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import catboost
from statsmodels.tools.eval_measures import rmse

def mape(y_true, y_pred, *, ignore_zeros=True):
    """
    Calculate Mean Absolute Percentage Error (MAPE).

    Parameters
    ----------
    y_true : array-like
        True values.
    y_pred : array-like
        Predicted values.
    ignore_zeros : bool, default True
        If True, excludes observations where y_true == 0.
        If False, raises an error when y_true contains zeros.

    Returns
    -------
    float
        MAPE value in percentage.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if y_true.shape != y_pred.shape:
        raise ValueError("y_true and y_pred must have the same shape")

    if ignore_zeros:
        mask = y_true != 0
        if not np.any(mask):
            raise ValueError("All y_true values are zero; MAPE is undefined")
        y_true = y_true[mask]
        y_pred = y_pred[mask]
    else:
        if np.any(y_true == 0):
            raise ValueError("y_true contains zeros; MAPE is undefined")

    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [13]:
train = pd.read_parquet("data/gold/train.parquet")
val = pd.read_parquet("data/gold/val.parquet")
test = pd.read_parquet("data/gold/test.parquet")

In [14]:
train["Money_spent"] = (
    train["Sum of кВт"]
    * (
        train["Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год"]
        + train["Average of Ціна ЕЕ грн. без ПДВ/кВт*год"]
    )
)

val["Money_spent"] = (
    val["Sum of кВт"]
    * (
        val["Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год"]
        + val["Average of Ціна ЕЕ грн. без ПДВ/кВт*год"]
    )
)

test["Money_spent"] = (
    test["Sum of кВт"]
    * (
        test["Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год"]
        + test["Average of Ціна ЕЕ грн. без ПДВ/кВт*год"]
    )
)

train.drop(columns=["Sum of кВт"], inplace=True)
val.drop(columns=["Sum of кВт"], inplace=True)
test.drop(columns=["Sum of кВт"], inplace=True)

In [15]:
train.to_parquet("data/money_calc/train.parquet", index=False)
val.to_parquet("data/money_calc/val.parquet", index=False)
test.to_parquet("data/money_calc/test.parquet", index=False)

In [16]:
y_col = "Money_spent"

In [17]:
features = train.columns.tolist()
features.remove("Money_spent")

X = train[features]
y = train[y_col]

In [18]:
lgb_model = lgb.LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, verbose=-1)
lgb_model.fit(X, y)

val_preds = lgb_model.predict(val[features])

val_rmse = rmse(val[y_col], val_preds)
val_mape = mape(val[y_col], val_preds)

print(f"RMSE: {val_rmse:.3f}")
print(f"MAPE: {val_mape:.3f}")

RMSE: 67.562
MAPE: 39.181


In [20]:
train[y_col].mean()

114.5737034377366